# Schema Migration Notebook

Materializes the operational layer described in [../DATABASE_DESIGN.md](../DATABASE_DESIGN.md) inside the existing BigQuery dataset.

**Idempotent**: every cell uses `CREATE TABLE IF NOT EXISTS`, `CREATE OR REPLACE VIEW`, or `MERGE`, so re-running the notebook is safe.

**Run order**:
1. Setup (env + BigQuery client)
2. DDL for new operational tables
3. Backfill `species` from distinct `tree` rows
4. Backfill `quarter_sections` + `qs_priority` from `quarter_section`
5. Backfill `trees_core` + `trees_features` from `tree`
6. Empty-create `users`, `service_requests`, `service_request_assignees`, `tree_inspections`
7. Cutover checks (new-table reads + analytics-source checks)
8. Assertions: row-count parity + column coverage

## 1. Setup

In [1]:
import os
from google.cloud import bigquery
from google.oauth2 import service_account

PROJECT_ID = os.environ.get('BQ_PROJECT_ID', 'mke-trees')
DATASET    = os.environ.get('BQ_DATASET', 'mke_tree_dataset')
LOCATION   = os.environ.get('BQ_LOCATION', 'us-central1')

SOURCE_TREE = os.environ.get('BQ_SOURCE_TREE_TABLE', f'{PROJECT_ID}.{DATASET}.tree')
SOURCE_QS   = os.environ.get('BQ_SOURCE_QS_TABLE',   f'{PROJECT_ID}.{DATASET}.quarter_section')

_KEY_CANDIDATES = [
    os.environ.get('GOOGLE_APPLICATION_CREDENTIALS', ''),
    os.path.join(os.path.dirname(os.path.abspath('.')), 'serviceAccountKey.json'),
    os.path.join(os.path.dirname(os.path.abspath('.')), 'database', 'serviceAccountKey.json'),
    os.path.abspath('serviceAccountKey.json'),
]
_key = next((p for p in _KEY_CANDIDATES if p and os.path.isfile(p)), None)

if _key:
    creds = service_account.Credentials.from_service_account_file(
        _key,
        scopes=['https://www.googleapis.com/auth/cloud-platform'],
    )
    client = bigquery.Client(credentials=creds, project=PROJECT_ID, location=LOCATION)
else:
    client = bigquery.Client(project=PROJECT_ID, location=LOCATION)

DS = f'`{PROJECT_ID}.{DATASET}`'
print(f'Project={PROJECT_ID} Dataset={DATASET} Location={LOCATION}')
print(f'Source tree table: {SOURCE_TREE}')
print(f'Source quarter_section table: {SOURCE_QS}')

def run(sql, label=None):
    """Run a single SQL statement and wait for it to complete."""
    if label:
        print(f'[run] {label}')
    job = client.query(sql, location=LOCATION)
    job.result()
    return job

c:\Users\edex\AppData\Local\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Project=mke-trees Dataset=mke_tree_dataset Location=us-central1
Source tree table: mke-trees.mke_tree_dataset.tree
Source quarter_section table: mke-trees.mke_tree_dataset.quarter_section


## 2. DDL — operational tables

Creates all 9 new tables. Idempotent via `CREATE TABLE IF NOT EXISTS`.

In [5]:
BASE_DDL = [
    f"""
    CREATE TABLE IF NOT EXISTS {DS}.users (
      user_id    STRING   NOT NULL,
      email      STRING   NOT NULL,
      role       STRING   NOT NULL,
      active     BOOL     NOT NULL,
      created_at TIMESTAMP NOT NULL,
      PRIMARY KEY (user_id) NOT ENFORCED
    )
    """,
    f"""
    CREATE TABLE IF NOT EXISTS {DS}.species (
      species_id      INT64  NOT NULL,
      species_code    STRING NOT NULL,
      common_name     STRING,
      scientific_name STRING,
      full_name       STRING,
      abbreviation    STRING,
      simple_species  STRING,
      genus           STRING,
      PRIMARY KEY (species_id) NOT ENFORCED
    )
    """,
    f"""
    CREATE TABLE IF NOT EXISTS {DS}.quarter_sections (
      qs_id        STRING NOT NULL,
      OBJECTID     INT64,
      SECTION_     STRING,
      TIER         STRING,
      `RANGE`      STRING,
      QTR          STRING,
      INDEX_ID     STRING,
      TOWN         STRING,
      QTRSEC_NW    STRING,
      QTRSEC_N     STRING,
      QTRSEC_NE    STRING,
      QTRSEC_W     STRING,
      QTRSEC_E     STRING,
      QTRSEC_SW    STRING,
      QTRSEC_S     STRING,
      QTRSEC_SE    STRING,
      QTRSEC_NAM   STRING,
      SHAPE_LENG   FLOAT64,
      district     STRING,
      last_pruned  INT64,
      pruning_cycle STRING,
      next_prune   FLOAT64,
      geometry     STRING,
      tree_count   INT64,
      created_at   TIMESTAMP NOT NULL,
      PRIMARY KEY (qs_id) NOT ENFORCED
    )
    CLUSTER BY district
    """,
]

PK_DDL = [
    f"ALTER TABLE {DS}.users ADD PRIMARY KEY (user_id) NOT ENFORCED",
    f"ALTER TABLE {DS}.species ADD PRIMARY KEY (species_id) NOT ENFORCED",
    f"ALTER TABLE {DS}.quarter_sections ADD PRIMARY KEY (qs_id) NOT ENFORCED",
]

DEPENDENT_DDL = [
    f"""
    CREATE TABLE IF NOT EXISTS {DS}.qs_priority (
      qs_id                     STRING NOT NULL,
      PS_critical               FLOAT64,
      PS_bottom90               FLOAT64,
      n                         FLOAT64,
      PS_background             FLOAT64,
      PS_composite              FLOAT64,
      Priority_Score_Normalized FLOAT64,
      ps_global                 FLOAT64,
      k                         FLOAT64,
      critical_weight           FLOAT64,
      PRIMARY KEY (qs_id) NOT ENFORCED,
      FOREIGN KEY (qs_id) REFERENCES {DS}.quarter_sections(qs_id) NOT ENFORCED
    )
    """,
    f"""
    CREATE TABLE IF NOT EXISTS {DS}.trees_core (
      tree_id    STRING NOT NULL,
      site_id    STRING,
      qs_id      STRING,
      species_id INT64,
      status     STRING NOT NULL,
      created_at TIMESTAMP NOT NULL,
      updated_at TIMESTAMP NOT NULL,

      address                          STRING,
      full_address                     STRING,
      street                           STRING,
      side                             STRING,
      site                             STRING,
      on_street                        STRING,
      closest_cross_street             STRING,
      side_of_street                   STRING,
      site_type                        STRING,
      direction_from_cross_street      FLOAT64,
      distance_from_cross_street       FLOAT64,
      latitude                         FLOAT64,
      longitude                        FLOAT64,
      district                         STRING,
      property_type                    STRING,
      census_block_id                  STRING,
      census_block_disadvantaged_area  BOOL,

      address_aerial                   STRING,
      street_aerial                    STRING,
      side_aerial                      STRING,
      site_aerial                      FLOAT64,
      on_street_aerial                 STRING,
      species_aerial                   STRING,
      dbh_aerial                       FLOAT64,
      inventory_date_aerial            STRING,
      condition_aerial                 STRING,
      alder_aerial                     FLOAT64,
      district_aerial                  STRING,
      property_type_aerial             STRING,
      pruning_cycle_aerial             FLOAT64,
      site_comments_aerial             STRING,
      _qs_str                          FLOAT64,

      dbh                              INT64,
      height                           FLOAT64,
      crown_width                      FLOAT64,
      crown_diameter_m                 FLOAT64,
      crown_width_m                    FLOAT64,
      crown_height_m                   FLOAT64,
      crown_area_sqm                   FLOAT64,
      crown_asymmetry                  FLOAT64,
      max_spread_ft                    FLOAT64,
      growing_space                    FLOAT64,
      tree_weight_kg                   FLOAT64,
      canopy_volume_index              INT64,
      slenderness                      FLOAT64,
      wood_density                     FLOAT64,

      condition                        FLOAT64,
      alder                            INT64,
      damage                           STRING,
      disease                          BOOL,
      canker                           BOOL,
      missing_or_dead                  STRING,
      reason_to_remove                 STRING,
      valuation_total                  FLOAT64,
      can_strike_building              BOOL,
      dist_closest_road_m              FLOAT64,
      dist_closest_building_m          FLOAT64,
      dist_closest_tree_m              FLOAT64,
      species_to_plant                 STRING,

      inventory_date                   STRING,
      pruning_cycle                    INT64,
      last_pruned                      FLOAT64,
      years_since_pruned               INT64,
      maintenance_deficit              INT64,
      estimated_age                    FLOAT64,
      age                              FLOAT64,
      age_method                       STRING,
      dbh_growth_yr                    FLOAT64,
      height_growth_yr                 FLOAT64,
      crown_growth_yr                  FLOAT64,
      site_last_changed_on             FLOAT64,
      site_comments                    STRING,

      PRIMARY KEY (tree_id) NOT ENFORCED,
      FOREIGN KEY (qs_id)      REFERENCES {DS}.quarter_sections(qs_id) NOT ENFORCED,
      FOREIGN KEY (species_id) REFERENCES {DS}.species(species_id) NOT ENFORCED
    )
    CLUSTER BY qs_id, status
    """,
    f"""
    CREATE TABLE IF NOT EXISTS {DS}.trees_features (
      tree_id                STRING NOT NULL,
      s_f                    FLOAT64,
      I_f_raw                FLOAT64,
      I_f                    FLOAT64,
      p_f                    FLOAT64,
      pruning_benefit_b      FLOAT64,
      a_p                    FLOAT64,
      k1                     INT64,
      k3                     INT64,
      risk_term_k1_I_f_p_f_b FLOAT64,
      age_term_k3_a_p        FLOAT64,
      priority_score         FLOAT64,
      PRIMARY KEY (tree_id) NOT ENFORCED,
      FOREIGN KEY (tree_id) REFERENCES {DS}.trees_core(tree_id) NOT ENFORCED
    )
    """,
    f"""
    CREATE TABLE IF NOT EXISTS {DS}.service_requests (
      service_request_id STRING NOT NULL,
      tree_id            STRING NOT NULL,
      request_type       STRING NOT NULL,
      priority           STRING NOT NULL,
      status             STRING NOT NULL,
      requested_at       TIMESTAMP NOT NULL,
      due_at             TIMESTAMP,
      completed_at       TIMESTAMP,
      notes              STRING,
      created_by         STRING NOT NULL,
      PRIMARY KEY (service_request_id) NOT ENFORCED,
      FOREIGN KEY (tree_id)    REFERENCES {DS}.trees_core(tree_id) NOT ENFORCED,
      FOREIGN KEY (created_by) REFERENCES {DS}.users(user_id) NOT ENFORCED
    )
    CLUSTER BY tree_id, status
    """,
    f"""
    CREATE TABLE IF NOT EXISTS {DS}.service_request_assignees (
      service_request_id STRING NOT NULL,
      user_id            STRING NOT NULL,
      assigned_at        TIMESTAMP NOT NULL,
      assigned_by        STRING,
      PRIMARY KEY (service_request_id, user_id) NOT ENFORCED,
      FOREIGN KEY (service_request_id) REFERENCES {DS}.service_requests(service_request_id) NOT ENFORCED,
      FOREIGN KEY (user_id)            REFERENCES {DS}.users(user_id) NOT ENFORCED,
      FOREIGN KEY (assigned_by)        REFERENCES {DS}.users(user_id) NOT ENFORCED
    )
    """,
    f"""
    CREATE TABLE IF NOT EXISTS {DS}.tree_inspections (
      inspection_id    STRING NOT NULL,
      tree_id          STRING NOT NULL,
      inspector_id     STRING NOT NULL,
      condition_rating FLOAT64,
      disease_flag     BOOL,
      notes            STRING,
      inspected_at     TIMESTAMP NOT NULL,
      PRIMARY KEY (inspection_id) NOT ENFORCED,
      FOREIGN KEY (tree_id)      REFERENCES {DS}.trees_core(tree_id) NOT ENFORCED,
      FOREIGN KEY (inspector_id) REFERENCES {DS}.users(user_id) NOT ENFORCED
    )
    """,
]


def run_allow_pk_exists(sql, label=None):
    try:
        return run(sql, label=label)
    except Exception as e:
        msg = str(e).lower()
        if (
            'already has primary key' in msg
            or 'already exists: constraint primary key' in msg
            or 'duplicate' in msg
        ):
            print(f'[skip] {label} (already present)')
            return None
        raise


for i, sql in enumerate(BASE_DDL, start=1):
    run(sql, label=f'BASE DDL {i}/{len(BASE_DDL)}')

for i, sql in enumerate(PK_DDL, start=1):
    run_allow_pk_exists(sql, label=f'PK DDL {i}/{len(PK_DDL)}')

for i, sql in enumerate(DEPENDENT_DDL, start=1):
    run(sql, label=f'DEP DDL {i}/{len(DEPENDENT_DDL)}')

print('DDL complete.')

[run] BASE DDL 1/3
[run] BASE DDL 2/3
[run] BASE DDL 3/3
[run] PK DDL 1/3
[skip] PK DDL 1/3 (already present)
[run] PK DDL 2/3
[run] PK DDL 3/3
[skip] PK DDL 3/3 (already present)
[run] DEP DDL 1/6
[run] DEP DDL 2/6
[run] DEP DDL 3/6
[run] DEP DDL 4/6
[run] DEP DDL 5/6
[run] DEP DDL 6/6
DDL complete.


## 3. Backfill `species`

Distinct species rows from current `tree`. `species_id` is generated with `ROW_NUMBER()` for stability.

In [7]:
# Ensure legacy species tables get required columns before MERGE references them.
for sql in [
    f"ALTER TABLE {DS}.species ADD COLUMN IF NOT EXISTS species_code STRING",
    f"ALTER TABLE {DS}.species ADD COLUMN IF NOT EXISTS common_name STRING",
    f"ALTER TABLE {DS}.species ADD COLUMN IF NOT EXISTS scientific_name STRING",
    f"ALTER TABLE {DS}.species ADD COLUMN IF NOT EXISTS full_name STRING",
    f"ALTER TABLE {DS}.species ADD COLUMN IF NOT EXISTS abbreviation STRING",
    f"ALTER TABLE {DS}.species ADD COLUMN IF NOT EXISTS simple_species STRING",
    f"ALTER TABLE {DS}.species ADD COLUMN IF NOT EXISTS genus STRING",
]:
    run(sql)

run(f"""
MERGE {DS}.species T USING (
  SELECT
    ROW_NUMBER() OVER (ORDER BY species_code) AS species_id,
    species_code,
    common_name,
    scientific_name,
    full_name,
    abbreviation,
    simple_species,
    genus
  FROM (
    SELECT DISTINCT
      COALESCE(species_code, abbreviation, scientific_name, species, 'UNKNOWN') AS species_code,
      ANY_VALUE(species)         AS common_name,
      ANY_VALUE(scientific_name) AS scientific_name,
      ANY_VALUE(full_name)       AS full_name,
      ANY_VALUE(abbreviation)    AS abbreviation,
      ANY_VALUE(simple_species)  AS simple_species,
      ANY_VALUE(genus)           AS genus
    FROM `{SOURCE_TREE}`
    GROUP BY COALESCE(species_code, abbreviation, scientific_name, species, 'UNKNOWN')
  )
) S
ON T.species_code = S.species_code
WHEN NOT MATCHED THEN INSERT (
  species_id, species_code, common_name, scientific_name, full_name, abbreviation, simple_species, genus
) VALUES (
  S.species_id, S.species_code, S.common_name, S.scientific_name, S.full_name, S.abbreviation, S.simple_species, S.genus
)
""", label='backfill species')

row = next(client.query(f'SELECT COUNT(*) AS n FROM {DS}.species').result())
print(f'species rows: {row.n}')

[run] backfill species
species rows: 152


## 4. Backfill `quarter_sections` + `qs_priority`

Splits the existing `quarter_section` table by destination.

In [8]:
run(f"""
MERGE {DS}.quarter_sections T USING (
  SELECT
    QTRSEC AS qs_id,
    OBJECTID,
    SECTION_,
    TIER,
    `RANGE`,
    QTR,
    INDEX_ID,
    TOWN,
    QTRSEC_NW, QTRSEC_N, QTRSEC_NE, QTRSEC_W, QTRSEC_E, QTRSEC_SW, QTRSEC_S, QTRSEC_SE, QTRSEC_NAM,
    SHAPE_LENG,
    district,
    last_pruned,
    pruning_cycle,
    next_prune,
    geometry,
    tree_count,
    CURRENT_TIMESTAMP() AS created_at
  FROM `{SOURCE_QS}`
  WHERE QTRSEC IS NOT NULL
) S
ON T.qs_id = S.qs_id
WHEN NOT MATCHED THEN INSERT (
  qs_id, OBJECTID, SECTION_, TIER, `RANGE`, QTR, INDEX_ID, TOWN,
  QTRSEC_NW, QTRSEC_N, QTRSEC_NE, QTRSEC_W, QTRSEC_E, QTRSEC_SW, QTRSEC_S, QTRSEC_SE, QTRSEC_NAM,
  SHAPE_LENG, district, last_pruned, pruning_cycle, next_prune, geometry, tree_count, created_at
) VALUES (
  S.qs_id, S.OBJECTID, S.SECTION_, S.TIER, S.`RANGE`, S.QTR, S.INDEX_ID, S.TOWN,
  S.QTRSEC_NW, S.QTRSEC_N, S.QTRSEC_NE, S.QTRSEC_W, S.QTRSEC_E, S.QTRSEC_SW, S.QTRSEC_S, S.QTRSEC_SE, S.QTRSEC_NAM,
  S.SHAPE_LENG, S.district, S.last_pruned, S.pruning_cycle, S.next_prune, S.geometry, S.tree_count, S.created_at
)
""", label='backfill quarter_sections')

run(f"""
MERGE {DS}.qs_priority T USING (
  SELECT
    QTRSEC AS qs_id,
    PS_critical, PS_bottom90, n, PS_background, PS_composite,
    Priority_Score_Normalized, ps_global, k, critical_weight
  FROM `{SOURCE_QS}`
  WHERE QTRSEC IS NOT NULL
) S
ON T.qs_id = S.qs_id
WHEN NOT MATCHED THEN INSERT ROW
""", label='backfill qs_priority')

for tbl in ('quarter_sections', 'qs_priority'):
    n = next(client.query(f'SELECT COUNT(*) AS n FROM {DS}.{tbl}').result()).n
    print(f'{tbl} rows: {n}')

[run] backfill quarter_sections
[run] backfill qs_priority
quarter_sections rows: 696
qs_priority rows: 696


## 5. Backfill `trees_core` + `trees_features`

Splits the existing `tree` table by destination, joining `species` to set `species_id`. `status` defaults to `'active'`; rows with `missing_or_dead` non-empty become `'dead'`; rows with `reason_to_remove` non-empty become `'removed'`.

In [9]:
run(f"""
MERGE {DS}.trees_core T USING (
  SELECT
    t.tree_row_id AS tree_id,
    t.site_id,
    t.quarter_section AS qs_id,
    s.species_id,
    CASE
      WHEN t.missing_or_dead IS NOT NULL AND TRIM(t.missing_or_dead) != '' THEN 'dead'
      WHEN t.reason_to_remove IS NOT NULL AND TRIM(t.reason_to_remove) != '' THEN 'removed'
      ELSE 'active'
    END AS status,
    CURRENT_TIMESTAMP() AS created_at,
    CURRENT_TIMESTAMP() AS updated_at,

    t.address, t.full_address, t.street, t.side, t.site, t.on_street, t.closest_cross_street,
    t.side_of_street, t.site_type, t.direction_from_cross_street, t.distance_from_cross_street,
    t.latitude, t.longitude, t.district, t.property_type, t.census_block_id, t.census_block_disadvantaged_area,

    t.address_aerial, t.street_aerial, t.side_aerial, t.site_aerial, t.on_street_aerial,
    t.species_aerial, t.dbh_aerial, t.inventory_date_aerial, t.condition_aerial, t.alder_aerial,
    t.district_aerial, t.property_type_aerial, t.pruning_cycle_aerial, t.site_comments_aerial, t._qs_str,

    t.dbh, t.height, t.crown_width, t.crown_diameter_m, t.crown_width_m, t.crown_height_m,
    t.crown_area_sqm, t.crown_asymmetry, t.max_spread_ft, t.growing_space, t.tree_weight_kg,
    t.canopy_volume_index, t.slenderness, t.wood_density,

    t.condition, t.alder, t.damage, t.disease, t.canker, t.missing_or_dead, t.reason_to_remove,
    t.valuation_total, t.can_strike_building, t.dist_closest_road_m, t.dist_closest_building_m,
    t.dist_closest_tree_m, t.species_to_plant,

    t.inventory_date, t.pruning_cycle, t.last_pruned, t.years_since_pruned, t.maintenance_deficit,
    t.estimated_age, t.age, t.age_method, t.dbh_growth_yr, t.height_growth_yr, t.crown_growth_yr,
    t.site_last_changed_on, t.site_comments
  FROM `{SOURCE_TREE}` t
  LEFT JOIN {DS}.species s
    ON s.species_code = COALESCE(t.species_code, t.abbreviation, t.scientific_name, t.species, 'UNKNOWN')
  WHERE t.tree_row_id IS NOT NULL
) S
ON T.tree_id = S.tree_id
WHEN NOT MATCHED THEN INSERT ROW
""", label='backfill trees_core')

run(f"""
MERGE {DS}.trees_features T USING (
  SELECT
    tree_row_id AS tree_id,
    s_f, I_f_raw, I_f, p_f, pruning_benefit_b, a_p, k1, k3,
    risk_term_k1_I_f_p_f_b, age_term_k3_a_p, priority_score
  FROM `{SOURCE_TREE}`
  WHERE tree_row_id IS NOT NULL
) S
ON T.tree_id = S.tree_id
WHEN NOT MATCHED THEN INSERT ROW
""", label='backfill trees_features')

for tbl in ('trees_core', 'trees_features'):
    n = next(client.query(f'SELECT COUNT(*) AS n FROM {DS}.{tbl}').result()).n
    print(f'{tbl} rows: {n}')

[run] backfill trees_core
[run] backfill trees_features
trees_core rows: 191074
trees_features rows: 191074


## 6. Empty-create operational entities

Already created by the DDL cell. This cell sanity-checks they exist and are empty.

In [10]:
for tbl in ('users', 'service_requests', 'service_request_assignees', 'tree_inspections'):
    n = next(client.query(f'SELECT COUNT(*) AS n FROM {DS}.{tbl}').result()).n
    print(f'{tbl}: {n} rows (expected 0 on first run)')

users: 0 rows (expected 0 on first run)
service_requests: 0 rows (expected 0 on first run)
service_request_assignees: 0 rows (expected 0 on first run)
tree_inspections: 0 rows (expected 0 on first run)


## 7. Cutover checks

This refactor targets the new tables directly. Compatibility views are optional temporary safety nets, not the default architecture.

Use this section to validate that key analytics fields and read paths resolve from the refactored schema before switching frontend/backend traffic fully to the new endpoints.

In [11]:
# Required: validate analytics fields from refactored tables
run(f"""
CREATE OR REPLACE VIEW {DS}.vw_qs_analytics_new AS
WITH tree_rollup AS (
  SELECT
    tc.qs_id,
    ANY_VALUE(sp.simple_species) AS top_species,
    COUNT(*) AS tree_count,
    AVG(CAST(tc.dbh AS FLOAT64)) AS avg_dbh,
    MAX(EXTRACT(YEAR FROM SAFE.PARSE_DATE('%Y-%m-%d', tc.inventory_date))) AS inspection_year
  FROM {DS}.trees_core tc
  LEFT JOIN {DS}.species sp USING (species_id)
  GROUP BY tc.qs_id
)
SELECT
  qs.qs_id,
  qs.district,
  tr.top_species,
  tr.tree_count,
  tr.avg_dbh,
  tr.inspection_year,
  CASE
    WHEN p.Priority_Score_Normalized >= 70 THEN 'Critical'
    WHEN p.Priority_Score_Normalized >= 50 THEN 'High'
    WHEN p.Priority_Score_Normalized >= 30 THEN 'Medium'
    ELSE 'Low'
  END AS priority_level,
  p.Priority_Score_Normalized
FROM {DS}.quarter_sections qs
LEFT JOIN {DS}.qs_priority p USING (qs_id)
LEFT JOIN tree_rollup tr USING (qs_id)
""", label='build refactor analytics view')

row = next(client.query(f"SELECT COUNT(*) AS n FROM {DS}.vw_qs_analytics_new").result())
print(f"vw_qs_analytics_new rows: {row.n}")

# Optional temporary compatibility views (transition only).
# Preferred final state is frontend/backend using refactored tables/endpoints directly.
CREATE_TEMP_VIEWS = False
if CREATE_TEMP_VIEWS:
    run(f"""
    CREATE OR REPLACE VIEW {DS}.vw_tree AS
    SELECT
      tc.tree_id AS tree_row_id,
      tc.qs_id AS quarter_section,
      tc.* EXCEPT(tree_id, qs_id, species_id, status, created_at, updated_at),
      sp.common_name AS species,
      sp.scientific_name,
      sp.full_name,
      sp.abbreviation,
      sp.species_code,
      sp.simple_species,
      sp.genus,
      tf.s_f, tf.I_f_raw, tf.I_f, tf.p_f, tf.pruning_benefit_b, tf.a_p, tf.k1, tf.k3,
      tf.risk_term_k1_I_f_p_f_b, tf.age_term_k3_a_p, tf.priority_score
    FROM {DS}.trees_core tc
    LEFT JOIN {DS}.trees_features tf USING (tree_id)
    LEFT JOIN {DS}.species sp USING (species_id)
    """, label='optional temp vw_tree')

    run(f"""
    CREATE OR REPLACE VIEW {DS}.vw_quarter_section AS
    SELECT
      qs.OBJECTID,
      qs.qs_id AS QTRSEC,
      qs.SECTION_, qs.TIER, qs.`RANGE`, qs.QTR, qs.INDEX_ID, qs.TOWN,
      qs.QTRSEC_NW, qs.QTRSEC_N, qs.QTRSEC_NE, qs.QTRSEC_W, qs.QTRSEC_E,
      qs.QTRSEC_SW, qs.QTRSEC_S, qs.QTRSEC_SE, qs.QTRSEC_NAM,
      qs.SHAPE_LENG, qs.district, qs.last_pruned, qs.pruning_cycle, qs.next_prune,
      qs.geometry, qs.tree_count,
      p.PS_critical, p.PS_bottom90, p.n, p.PS_background, p.PS_composite,
      p.Priority_Score_Normalized, p.ps_global, p.k, p.critical_weight
    FROM {DS}.quarter_sections qs
    LEFT JOIN {DS}.qs_priority p USING (qs_id)
    """, label='optional temp vw_quarter_section')

    print('Temporary compatibility views created.')

[run] build refactor analytics view
vw_qs_analytics_new rows: 696


## 8. Assertions — row-count parity + column coverage

Fails loudly if backfill missed rows or if any source column was forgotten. Run after cutover checks.

In [12]:
src_tree_n = next(client.query(f'SELECT COUNT(*) AS n FROM `{SOURCE_TREE}`').result()).n
src_qs_n   = next(client.query(f'SELECT COUNT(*) AS n FROM `{SOURCE_QS}`').result()).n
core_n     = next(client.query(f'SELECT COUNT(*) AS n FROM {DS}.trees_core').result()).n
feat_n     = next(client.query(f'SELECT COUNT(*) AS n FROM {DS}.trees_features').result()).n
qs_n       = next(client.query(f'SELECT COUNT(*) AS n FROM {DS}.quarter_sections').result()).n
prio_n     = next(client.query(f'SELECT COUNT(*) AS n FROM {DS}.qs_priority').result()).n

print(f'tree            source rows: {src_tree_n}')
print(f'trees_core      target rows: {core_n}')
print(f'trees_features  target rows: {feat_n}')
print(f'quarter_section source rows: {src_qs_n}')
print(f'quarter_sections target rows: {qs_n}')
print(f'qs_priority     target rows: {prio_n}')

assert core_n == src_tree_n, f'trees_core row drift: {core_n} != {src_tree_n}'
assert feat_n == src_tree_n, f'trees_features row drift: {feat_n} != {src_tree_n}'
assert qs_n   == src_qs_n,   f'quarter_sections row drift: {qs_n} != {src_qs_n}'
assert prio_n == src_qs_n,   f'qs_priority row drift: {prio_n} != {src_qs_n}'
print('Row parity OK.')


tree            source rows: 191074
trees_core      target rows: 191074
trees_features  target rows: 191074
quarter_section source rows: 696
quarter_sections target rows: 696
qs_priority     target rows: 696
Row parity OK.


In [13]:
TREE_TARGETS = {
    'trees_core': {
        'site_id', 'quarter_section', 'address', 'street', 'side', 'site', 'on_street',
        'dbh', 'latitude', 'longitude', 'inventory_date', 'species_to_plant', 'condition',
        'alder', 'closest_cross_street', 'crown_width', 'damage', 'direction_from_cross_street',
        'disease', 'distance_from_cross_street', 'district', 'growing_space', 'height',
        'property_type', 'pruning_cycle', 'reason_to_remove', 'side_of_street', 'site_type',
        'valuation_total', 'site_last_changed_on', 'site_comments', 'census_block_disadvantaged_area',
        'census_block_id', 'dbh_growth_yr', 'height_growth_yr', 'crown_growth_yr', 'canker',
        'slenderness', 'wood_density', 'tree_weight_kg', 'canopy_volume_index', 'full_address',
        'last_pruned', 'estimated_age', 'age_method', 'address_aerial', 'street_aerial',
        'side_aerial', 'site_aerial', 'on_street_aerial', 'species_aerial', 'dbh_aerial',
        'inventory_date_aerial', 'condition_aerial', 'alder_aerial', 'district_aerial',
        'property_type_aerial', 'pruning_cycle_aerial', 'site_comments_aerial', '_qs_str',
        'max_spread_ft', 'dist_closest_road_m', 'dist_closest_building_m', 'dist_closest_tree_m',
        'crown_area_sqm', 'crown_width_m', 'crown_height_m', 'crown_asymmetry', 'missing_or_dead',
        'crown_diameter_m', 'age', 'years_since_pruned', 'tree_row_id', 'maintenance_deficit',
        'can_strike_building',
    },
    'species': {'species', 'scientific_name', 'full_name', 'abbreviation', 'species_code',
                'simple_species', 'genus'},
    'trees_features': {'s_f', 'I_f_raw', 'I_f', 'p_f', 'pruning_benefit_b', 'a_p', 'k1', 'k3',
                       'risk_term_k1_I_f_p_f_b', 'age_term_k3_a_p', 'priority_score'},
}
QS_TARGETS = {
    'quarter_sections': {'OBJECTID', 'QTRSEC', 'SECTION_', 'TIER', 'RANGE', 'QTR', 'INDEX_ID',
                         'TOWN', 'QTRSEC_NW', 'QTRSEC_N', 'QTRSEC_NE', 'QTRSEC_W', 'QTRSEC_E',
                         'QTRSEC_SW', 'QTRSEC_S', 'QTRSEC_SE', 'QTRSEC_NAM', 'SHAPE_LENG',
                         'district', 'last_pruned', 'pruning_cycle', 'next_prune', 'geometry',
                         'tree_count'},
    'qs_priority':      {'PS_critical', 'PS_bottom90', 'n', 'PS_background', 'PS_composite',
                         'Priority_Score_Normalized', 'ps_global', 'k', 'critical_weight'},
}

covered_tree = set().union(*TREE_TARGETS.values())
covered_qs   = set().union(*QS_TARGETS.values())

tree_cols_q = client.query(
    f"""
    SELECT column_name FROM `{PROJECT_ID}.{DATASET}.INFORMATION_SCHEMA.COLUMNS`
    WHERE table_name = '{SOURCE_TREE.split(".")[-1]}'
    """
).result()
qs_cols_q = client.query(
    f"""
    SELECT column_name FROM `{PROJECT_ID}.{DATASET}.INFORMATION_SCHEMA.COLUMNS`
    WHERE table_name = '{SOURCE_QS.split(".")[-1]}'
    """
).result()

tree_cols = {r.column_name for r in tree_cols_q}
qs_cols   = {r.column_name for r in qs_cols_q}

missing_tree = tree_cols - covered_tree
missing_qs   = qs_cols   - covered_qs
extra_tree   = covered_tree - tree_cols
extra_qs     = covered_qs   - qs_cols

print('--- column coverage ---')
print(f'tree source columns:           {len(tree_cols)}')
print(f'tree mapped columns:           {len(covered_tree)}')
print(f'tree missing in mapping:       {sorted(missing_tree)}')
print(f'tree extra in mapping:         {sorted(extra_tree)}')
print(f'quarter_section source cols:   {len(qs_cols)}')
print(f'quarter_section mapped cols:   {len(covered_qs)}')
print(f'quarter_section missing:       {sorted(missing_qs)}')
print(f'quarter_section extra:         {sorted(extra_qs)}')

assert not missing_tree, f'tree columns not mapped: {missing_tree}'
assert not missing_qs,   f'quarter_section columns not mapped: {missing_qs}'
print('Column coverage OK.')

--- column coverage ---
tree source columns:           93
tree mapped columns:           93
tree missing in mapping:       []
tree extra in mapping:         []
quarter_section source cols:   33
quarter_section mapped cols:   33
quarter_section missing:       []
quarter_section extra:         []
Column coverage OK.
